# 🎯 Feature Selection
## Which features should I select for basin generalisation?

**R&D Notebook** — Correlation analysis, permutation importance, recursive feature elimination, L1 probing, and domain-generalisation-specific feature selection.

**Depends on**: Feature Engineering notebook (uses enhanced feature set).

---

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
QUICK_MODE = True
TARGET_BASIN = 'SI'
EPOCHS = 3 if QUICK_MODE else 15
RFE_EPOCHS = 2 if QUICK_MODE else 8  # Reduced epochs for RFE iterations
USE_3D = not QUICK_MODE
USE_ENV = True
BATCH_SIZE = 64 if QUICK_MODE else 32

In [ ]:
import os, glob, math, time, warnings, copy
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from netCDF4 import Dataset as NC4Dataset
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics import f1_score, precision_recall_fscore_support

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
matplotlib.rcParams['figure.dpi'] = 120

PROJECT_ROOT = Path('/Users/thiruanand/MLfTCC')
DATA_ROOT    = PROJECT_ROOT / 'Notebooks' / 'data' / 'tropicyclonenet' / 'TCND_test'
BASINS = ['EP', 'NA', 'NI', 'SI', 'SP', 'WP']
BASIN_FULL = {'EP':'Eastern Pacific','NA':'North Atlantic','NI':'North Indian',
              'SI':'South Indian','SP':'South Pacific','WP':'Western Pacific'}
INT_NAMES = ['Weakening','Slow Weakening','Steady','Intensification']  # 4 classes from dataset (0-3)
RI_CLASS = 3  # Intensification is the highest class (class 3)

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42; torch.manual_seed(SEED); np.random.seed(SEED)

print(f"✓ Setup | Device: {DEVICE} | Quick: {QUICK_MODE}")

## 1. Data Loading & Enhanced Feature Computation
Reuses data pipeline + enhanced features from Feature Engineering notebook.

In [ ]:
# ── Data loading functions (from base notebook) ──
def scan_files(data_root, basins, subdir, ext):
    idx = {}
    for b in basins:
        d = data_root / subdir / b
        if not d.exists(): continue
        for f in sorted(d.rglob(f'*{ext}')):
            storm = f.parent.name if ext=='.npy' else f.stem.split('_')[1]
            dt = f.stem if ext=='.npy' else f.stem.split('_')[2]
            idx[(b, storm, dt)] = str(f)
    return idx

def build_env_vector(env_dict):
    def to_scalar(x):
        return np.float32(np.argmax(x)) if isinstance(x, np.ndarray) else np.float32(max(x, 0))
    return np.concatenate([
        np.array(env_dict.get('month', np.zeros(12)), dtype=np.float32),
        np.array(env_dict.get('area', np.zeros(6)), dtype=np.float32),
        np.array(env_dict.get('intensity_class', np.zeros(6)), dtype=np.float32),
        np.array([env_dict.get('wind', 0)], dtype=np.float32),
        np.array([env_dict.get('move_velocity', 0)], dtype=np.float32),
        np.array(env_dict.get('location_long', np.zeros(36)), dtype=np.float32),
        np.array(env_dict.get('location_lat', np.zeros(12)), dtype=np.float32),
        np.array([to_scalar(env_dict.get('history_direction12',-1))], dtype=np.float32),
        np.array([to_scalar(env_dict.get('history_direction24',-1))], dtype=np.float32),
        np.array([to_scalar(env_dict.get('history_inte_change24',-1))], dtype=np.float32),
    ])

def load_data3d(nc_path):
    with NC4Dataset(nc_path, 'r') as nc:
        u = np.array(nc.variables['u'][:], dtype=np.float32)
        v = np.array(nc.variables['v'][:], dtype=np.float32)
        z_geo = np.array(nc.variables['z'][:], dtype=np.float32)
        sst_var = nc.variables['sst']
        sst_raw = np.ma.filled(sst_var[:], fill_value=np.nan).astype(np.float32)
    chs = [u[0,i] for i in range(4)] + [v[0,i] for i in range(4)] + [z_geo[0,i] for i in range(4)]
    chs.append(sst_raw[0] if sst_raw.ndim==3 else sst_raw)
    data_3d = np.stack(chs, axis=0)
    for c in range(13):
        ch = data_3d[c]; valid = ch[~np.isnan(ch)]
        if len(valid) > 0:
            mu, std = valid.mean(), max(valid.std(), 1e-6)
            ch[np.isnan(ch)] = mu; data_3d[c] = (ch - mu) / std
        else: data_3d[c] = np.zeros_like(ch)
    return data_3d

def compute_labeled_dataset(data_root, basins):
    """Load all samples with CORRECT column order and targets from Env-Data."""
    records = []
    d3d_idx = scan_files(data_root, basins, 'Data3D', '.nc')
    env_idx = scan_files(data_root, basins, 'Env-Data', '.npy')
    # Correct column order (from src/data/dataset.py):
    # ID | FLAG | LAT_norm | LONG_norm | WND_norm | PRES_norm | YYYYMMDDHH | Name
    TCND_COLS = ['ID', 'FLAG', 'LAT_norm', 'LONG_norm', 'WND_norm', 'PRES_norm', 'YYYYMMDDHH', 'Name']
    for basin in basins:
        d = data_root / 'Data1D' / basin / 'test'
        if not d.exists(): continue
        for f in sorted(d.glob('*.txt')):
            try:
                df = pd.read_csv(f, sep='\t', header=None, names=TCND_COLS)
                stem = f.stem
                year = stem[len(basin):len(basin)+4]
                tc_name = stem[len(basin)+4+3:]  # skip 'BST'
                for _, r in df.iterrows():
                    dt = str(int(r['YYYYMMDDHH']))
                    storm = tc_name
                    d3d_p = d3d_idx.get((basin,storm.upper(),dt)) or d3d_idx.get((basin,storm,dt))
                    env_p = env_idx.get((basin,storm.upper(),dt)) or env_idx.get((basin,storm,dt))
                    if not (d3d_p and env_p): continue
                    # Read TARGETS from Env-Data (NOT computed from wind change)
                    env_d = np.load(env_p, allow_pickle=True).item()
                    y_int = int(np.asarray(env_d.get('future_inte_change24', 2)).ravel()[0])
                    y_dir = int(np.asarray(env_d.get('future_direction24', 0)).ravel()[0])
                    if y_int < 0: y_int = 2  # sentinel -1 → steady
                    if y_dir < 0: y_dir = 0
                    y_int = min(y_int, 4)
                    y_dir = min(y_dir, 7)
                    # data1d: [LONG, LAT, PRES, WND] (matching src/data/dataset.py)
                    records.append({'basin':basin,'storm':storm,'datetime':dt,
                        'data1d':np.array([r['LONG_norm'],r['LAT_norm'],r['PRES_norm'],r['WND_norm']],dtype=np.float32),
                        'data3d_path':d3d_p,'env_path':env_p,'y_intensity':y_int,'y_direction':y_dir})
            except: pass
    return records

# ── Enhanced features (from Feature Engineering notebook) ──
def compute_enhanced_features(sample):
    lon, lat, pres, wnd = sample['data1d']
    abs_lat = abs(lat); lat_rad = abs_lat * np.pi / 2
    coriolis_enhanced = 2 * 7.2921e-5 * np.sin(np.clip(lat_rad, 0, np.pi/2))
    sst_proxy = wnd * 0.5; shear = abs(pres - wnd)
    return {
        'wind_pres_ratio': float(wnd / max(abs(pres), 1e-6)),
        'wind_pres_diff': float(wnd - pres),
        'abs_lat': float(abs_lat),
        'coriolis_enhanced': float(coriolis_enhanced),
        'coriolis_x_wind': float(coriolis_enhanced * max(wnd, 0)),
        'sst_proxy': float(sst_proxy),
        'sst_bin': float(int(np.clip(sst_proxy * 5, 0, 4))),
        'shear': float(shear),
        'shear_int_interaction': float(shear * wnd),
        'favourable_shear': float(shear < 0.15),
        'translation_proxy': float(abs(lon) * 0.1),
        'mpi_deficit': float(max(0, sst_proxy * 2 - shear) - wnd),
        'intensification_potential': float(max(0, max(0, sst_proxy * 2 - shear) - wnd)),
        'wnd_squared': float(wnd ** 2),
        'pres_cubed': float(pres ** 3),
    }

ENHANCED_FEATURE_NAMES = list(compute_enhanced_features(({'data1d': np.zeros(4)})).keys())

print("Building labeled dataset...")
t0 = time.time()
ALL_SAMPLES = compute_labeled_dataset(DATA_ROOT, BASINS)
print(f"✓ {len(ALL_SAMPLES):,} samples in {time.time()-t0:.1f}s")

## 2. Feature Correlation Analysis
Correlation matrix of all scalar features with hierarchical clustering.

In [ ]:
# Build full scalar feature matrix
DATA1D_NAMES = ['LONG_norm', 'LAT_norm', 'PRES_norm', 'WND_norm']
ORIG_PHYS_NAMES = ['sst_anom', 'wind_shear', 'coriolis', 'mpi_proxy',
                   'bl_moisture', 'outflow_temp', 'steering', 'current_int']
ALL_FEATURE_NAMES = DATA1D_NAMES + ORIG_PHYS_NAMES + ENHANCED_FEATURE_NAMES

feat_rows = []
for s in ALL_SAMPLES:
    lon, lat, pres, wnd = s['data1d']
    # Original physics
    sst_anom = wnd * 0.5; wind_shear = abs(pres - wnd)
    lat_rad = abs(lat) * 0.3
    coriolis = 2 * 7.2921e-5 * np.sin(np.clip(lat_rad, -np.pi/2, np.pi/2))
    mpi_proxy = max(0, sst_anom * 2 - wind_shear)
    bl_moisture = max(0, 1.0 - abs(pres))
    outflow_temp = -0.5 * abs(lat); steering = lon * 0.1; current_int = wnd
    orig_phys = [sst_anom, wind_shear, coriolis, mpi_proxy, bl_moisture, outflow_temp, steering, current_int]
    # Enhanced
    enh = compute_enhanced_features(s)
    enh_vals = [enh[k] for k in ENHANCED_FEATURE_NAMES]
    row = list(s['data1d']) + orig_phys + enh_vals
    feat_rows.append(row)

X_all = np.array(feat_rows)
feat_df = pd.DataFrame(X_all, columns=ALL_FEATURE_NAMES)
feat_df['y_intensity'] = [s['y_intensity'] for s in ALL_SAMPLES]
feat_df['basin'] = [s['basin'] for s in ALL_SAMPLES]

# Correlation matrix
corr = feat_df[ALL_FEATURE_NAMES].corr()

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(corr, annot=False, cmap='RdBu_r', center=0, ax=ax, vmin=-1, vmax=1,
            xticklabels=True, yticklabels=True)
ax.set_title('Feature Correlation Matrix (with hierarchical clustering)', fontweight='bold')
plt.xticks(fontsize=7, rotation=45, ha='right'); plt.yticks(fontsize=7)
plt.tight_layout(); plt.show()

print(f"Feature matrix shape: {X_all.shape}")
print(f"Total features: {len(ALL_FEATURE_NAMES)}")

## 3. Variance & Redundancy Filtering
Identify near-zero-variance and highly correlated features.

In [ ]:
# Variance analysis
variances = feat_df[ALL_FEATURE_NAMES].var()
print("Feature Variances (sorted ascending):")
for name, var in variances.sort_values().items():
    flag = " ⚠️ LOW" if var < 0.001 else ""
    print(f"  {name:<30} var={var:.6f}{flag}")

low_var_features = variances[variances < 0.001].index.tolist()
print(f"\nLow-variance features (var < 0.001): {low_var_features}")

# Highly correlated pairs
print("\nHighly correlated pairs (|r| > 0.90):")
high_corr_pairs = []
for i, f1 in enumerate(ALL_FEATURE_NAMES):
    for j, f2 in enumerate(ALL_FEATURE_NAMES):
        if i < j and abs(corr.loc[f1, f2]) > 0.90:
            high_corr_pairs.append((f1, f2, corr.loc[f1, f2]))
            print(f"  {f1} ↔ {f2}: r={corr.loc[f1, f2]:.3f}")

if not high_corr_pairs:
    print("  None found.")

# Redundancy candidates
redundant_candidates = set()
for f1, f2, r in high_corr_pairs:
    # Keep the one with higher variance
    if variances[f1] < variances[f2]:
        redundant_candidates.add(f1)
    else:
        redundant_candidates.add(f2)

print(f"\nSuggested removals (redundant): {redundant_candidates}")

## 4. Model-Based Importance (Permutation)
Train baseline, then measure drop in accuracy when each feature group is permuted.

In [ ]:
# ── Model & Dataset infrastructure ──
ORIG_PHYS_DIM = 8
NEW_PHYS_DIM = ORIG_PHYS_DIM + len(ENHANCED_FEATURE_NAMES)

class TCNDDatasetSelectable(Dataset):
    """Dataset with selectable physics features via a mask."""
    def __init__(self, samples, use_3d=True, use_env=True, feature_mask=None):
        self.samples = samples
        self.use_3d, self.use_env = use_3d, use_env
        self.feature_mask = feature_mask  # Boolean mask over physics features
        self.basin_to_idx = {b:i for i,b in enumerate(BASINS)}
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        item = {
            'data_1d': torch.tensor(s['data1d'], dtype=torch.float32),
            'y_intensity': torch.tensor(s['y_intensity'], dtype=torch.long),
            'y_direction': torch.tensor(s['y_direction'], dtype=torch.long),
            'basin_idx': torch.tensor(self.basin_to_idx[s['basin']], dtype=torch.long),
        }
        if self.use_3d:
            item['data_3d'] = torch.tensor(load_data3d(s['data3d_path']), dtype=torch.float32)
        else:
            item['data_3d'] = torch.zeros(13, 81, 81)
        if self.use_env:
            env = np.load(s['env_path'], allow_pickle=True).item()
            item['env_data'] = torch.tensor(build_env_vector(env), dtype=torch.float32)
        else:
            item['env_data'] = torch.zeros(77)
        # All physics features (original + enhanced)
        lon, lat, pres, wnd = s['data1d']
        sst_anom = wnd * 0.5; wind_shear = abs(pres - wnd)
        lat_rad = abs(lat) * 0.3
        coriolis = 2 * 7.2921e-5 * np.sin(np.clip(lat_rad, -np.pi/2, np.pi/2))
        mpi_proxy = max(0, sst_anom * 2 - wind_shear)
        bl_moisture = max(0, 1.0 - abs(pres))
        outflow_temp = -0.5 * abs(lat); steering = lon * 0.1; current_int = wnd
        orig = [sst_anom, wind_shear, coriolis, mpi_proxy, bl_moisture, outflow_temp, steering, current_int]
        enh = compute_enhanced_features(s)
        enh_vals = [enh[k] for k in ENHANCED_FEATURE_NAMES]
        all_phys = np.array(orig + enh_vals, dtype=np.float32)
        if self.feature_mask is not None:
            all_phys = all_phys[self.feature_mask]
        item['phys_features'] = torch.tensor(all_phys, dtype=torch.float32)
        return item

print(f"Total physics features: {NEW_PHYS_DIM} (8 original + {len(ENHANCED_FEATURE_NAMES)} enhanced)")

In [ ]:
# ── Model architecture ──
class ConvBnRelu(nn.Sequential):
    def __init__(self, in_c, out_c, k=3, s=1, p=1):
        super().__init__(nn.Conv2d(in_c,out_c,k,stride=s,padding=p,bias=False),
                         nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))
class SpatialEncoder(nn.Module):
    def __init__(self, in_ch=13, embed_dim=128):
        super().__init__()
        self.net = nn.Sequential(ConvBnRelu(in_ch,32,k=7,s=2,p=3), ConvBnRelu(32,64,k=3,s=2,p=1),
            ConvBnRelu(64,128,k=3,s=2,p=1), ConvBnRelu(128,256,k=3,s=2,p=1),
            nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.proj = nn.Sequential(nn.Linear(256, embed_dim), nn.LayerNorm(embed_dim))
    def forward(self, x): return self.proj(self.net(x))
class TrackEncoder(nn.Module):
    def __init__(self, in_dim=4, embed_dim=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim,64), nn.LayerNorm(64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64,embed_dim), nn.LayerNorm(embed_dim))
    def forward(self, x): return self.net(x)
class EnvEncoder(nn.Module):
    def __init__(self, in_dim=77, embed_dim=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim,128), nn.LayerNorm(128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128,embed_dim), nn.LayerNorm(embed_dim))
    def forward(self, x): return self.net(x)
class PhysicsEncoder(nn.Module):
    def __init__(self, in_dim=8, phys_dim=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim,64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(64,phys_dim), nn.LayerNorm(phys_dim))
    def forward(self, x): return self.net(x)
class MultimodalBackbone(nn.Module):
    def __init__(self, spatial_embed=128, track_embed=32, env_embed=64,
                 phys_dim=32, phys_in_dim=8, final_dim=128, use_3d=True, use_env=True):
        super().__init__()
        self.use_3d, self.use_env, self.phys_dim = use_3d, use_env, phys_dim
        self.spatial_enc = SpatialEncoder(embed_dim=spatial_embed) if use_3d else None
        self.track_enc = TrackEncoder(embed_dim=track_embed)
        self.env_enc = EnvEncoder(embed_dim=env_embed) if use_env else None
        self.phys_enc = PhysicsEncoder(in_dim=phys_in_dim, phys_dim=phys_dim)
        fused_in = track_embed + (spatial_embed if use_3d else 0) + (env_embed if use_env else 0)
        self.projector = nn.Sequential(nn.Linear(fused_in, final_dim*2), nn.LayerNorm(final_dim*2),
            nn.GELU(), nn.Dropout(0.1), nn.Linear(final_dim*2, final_dim), nn.LayerNorm(final_dim))
        self.env_dim = final_dim - phys_dim; self.final_dim = final_dim
        self.phys_align = nn.Linear(phys_dim, phys_dim)
    def forward(self, batch):
        parts = []
        if self.use_3d: parts.append(self.spatial_enc(batch['data_3d']))
        parts.append(self.track_enc(batch['data_1d']))
        if self.use_env: parts.append(self.env_enc(batch['env_data']))
        z_full = self.projector(torch.cat(parts, dim=-1))
        z_env = z_full[:, :self.env_dim]; z_phys_raw = z_full[:, self.env_dim:]
        z_phys = z_phys_raw + self.phys_align(self.phys_enc(batch['phys_features']))
        return {'z':torch.cat([z_env, z_phys], dim=-1), 'z_phys':z_phys, 'z_phys_raw':z_phys_raw, 'z_env':z_env}
class TaskHeads(nn.Module):
    def __init__(self, in_dim=128, n_int=4, n_dir=8):
        super().__init__()
        self.int_head = nn.Sequential(nn.Linear(in_dim,64), nn.GELU(), nn.Dropout(0.1), nn.Linear(64,n_int))
        self.dir_head = nn.Sequential(nn.Linear(in_dim,64), nn.GELU(), nn.Dropout(0.1), nn.Linear(64,n_dir))
    def forward(self, z): return self.int_head(z), self.dir_head(z)
class TCModelFS(nn.Module):
    def __init__(self, use_3d=True, use_env=True, phys_dim=32, phys_in_dim=8, final_dim=128):
        super().__init__()
        self.backbone = MultimodalBackbone(use_3d=use_3d, use_env=use_env,
            phys_dim=phys_dim, phys_in_dim=phys_in_dim, final_dim=final_dim)
        self.heads = TaskHeads(in_dim=final_dim)
    def forward(self, batch):
        feat = self.backbone(batch)
        li, ld = self.heads(feat['z'])
        return {**feat, 'logits_intensity':li, 'logits_direction':ld}

# ── Training utilities ──
def task_loss(li, ld, yi, yd): return F.cross_entropy(li, yi) + 0.5 * F.cross_entropy(ld, yd)
class ERM:
    def compute_loss(self, batches, model):
        losses = [task_loss(model(b)['logits_intensity'], model(b)['logits_direction'],
                            b['y_intensity'], b['y_direction']) for b in batches.values()]
        erm = torch.stack(losses).mean()
        return erm, {}
def make_lobo_split(all_samples, target_basin, val_frac=0.15):
    source = [s for s in all_samples if s['basin'] != target_basin]
    target = [s for s in all_samples if s['basin'] == target_basin]
    np.random.shuffle(source); n_val = int(len(source)*val_frac)
    return source[n_val:], source[:n_val], target
def make_per_basin_loaders(samples, bs, ds_cls, **kw):
    by_basin = defaultdict(list)
    for s in samples: by_basin[s['basin']].append(s)
    return {b: DataLoader(ds_cls(samps, **kw), batch_size=min(bs,len(samps)), shuffle=True, drop_last=True, num_workers=0)
            for b, samps in by_basin.items()}
def train_epoch(model, method, loaders, opt, dev):
    model.train(); iters = {b:iter(l) for b,l in loaders.items()}; nb = max(len(l) for l in loaders.values()); loss=0
    for _ in range(nb):
        batches = {}
        for b,it in iters.items():
            try: batch=next(it)
            except StopIteration: iters[b]=iter(loaders[b]); batch=next(iters[b])
            batches[b]={k:v.to(dev) if isinstance(v,torch.Tensor) else v for k,v in batch.items()}
        opt.zero_grad(); l,_=method.compute_loss(batches,model); l.backward()
        nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); loss+=l.item()
    return loss/nb
def evaluate(model, loader, dev):
    model.eval(); pi,li=[],[]
    with torch.no_grad():
        for b in loader:
            b={k:v.to(dev) if isinstance(v,torch.Tensor) else v for k,v in b.items()}
            pi.extend(model(b)['logits_intensity'].argmax(-1).cpu().numpy())
            li.extend(b['y_intensity'].cpu().numpy())
    pi,li=np.array(pi),np.array(li)
    prec,rec,f1,_=precision_recall_fscore_support(li,pi,labels=[RI_CLASS],zero_division=0)
    return {'acc':float((pi==li).mean()),'wf1':float(f1_score(li,pi,average='weighted',zero_division=0)),
            'ri_f1':float(f1[0])}

def run_fs_experiment(name, all_samples, target, feature_mask, epochs, use_3d, use_env, bs):
    n_feats = int(feature_mask.sum()) if feature_mask is not None else NEW_PHYS_DIM
    phys_dim = min(32, max(8, n_feats)); final_dim = 128 if use_3d else (96 if use_env else 48)
    train_s,val_s,test_s = make_lobo_split(all_samples, target)
    kw = dict(use_3d=use_3d, use_env=use_env, feature_mask=feature_mask)
    tl = make_per_basin_loaders(train_s, bs, TCNDDatasetSelectable, **kw)
    vl = DataLoader(TCNDDatasetSelectable(val_s,**kw), batch_size=bs, num_workers=0)
    el = DataLoader(TCNDDatasetSelectable(test_s,**kw), batch_size=bs, num_workers=0)
    model = TCModelFS(use_3d=use_3d, use_env=use_env, phys_dim=phys_dim, phys_in_dim=n_feats, final_dim=final_dim).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    method = ERM(); best_wf1=-1; best_state=None
    for ep in range(1,epochs+1):
        t0=time.time(); loss=train_epoch(model,method,tl,opt,DEVICE)
        vr=evaluate(model,vl,DEVICE)
        if vr['wf1']>best_wf1: best_wf1=vr['wf1']; best_state=copy.deepcopy(model.state_dict())
        print(f"  [{name}] Ep {ep}/{epochs} | loss={loss:.4f} | acc={vr['acc']:.3f} | wF1={vr['wf1']:.3f} | {time.time()-t0:.1f}s")
    if best_state: model.load_state_dict(best_state)
    tr = evaluate(model,el,DEVICE)
    print(f"  → Test: acc={tr['acc']:.3f} | wF1={tr['wf1']:.3f} | RI-F1={tr['ri_f1']:.3f}")
    return {'name':name, 'n_feats':n_feats, **tr}

print("✓ Infrastructure defined")

In [ ]:
# Permutation importance: train once with all features, then permute each group
ALL_PHYS_NAMES = ORIG_PHYS_NAMES + ENHANCED_FEATURE_NAMES
print(f"Physics feature vector: {len(ALL_PHYS_NAMES)} features")

# Train baseline with all features
print("\nTraining baseline with all features...")
base_result = run_fs_experiment('All Features', ALL_SAMPLES, TARGET_BASIN,
    feature_mask=None, epochs=EPOCHS, use_3d=USE_3D, use_env=USE_ENV, bs=BATCH_SIZE)
base_acc = base_result['acc']

# Leave-one-out importance: drop each feature one at a time
print("\nLeave-one-out feature importance...")
importances = []
for drop_idx, fname in enumerate(ALL_PHYS_NAMES):
    mask = np.ones(NEW_PHYS_DIM, dtype=bool)
    mask[drop_idx] = False
    r = run_fs_experiment(f'-{fname}', ALL_SAMPLES, TARGET_BASIN,
        feature_mask=mask, epochs=RFE_EPOCHS, use_3d=USE_3D, use_env=USE_ENV, bs=BATCH_SIZE)
    drop_in_acc = base_acc - r['acc']
    importances.append({'feature': fname, 'acc_drop': drop_in_acc, 'acc_without': r['acc'],
                        'wf1_without': r['wf1'], 'ri_f1_without': r['ri_f1']})

imp_df = pd.DataFrame(importances).sort_values('acc_drop', ascending=False)
display(imp_df.round(4))

In [ ]:
# Visualise importance
fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#2ecc71' if d > 0 else '#e74c3c' for d in imp_df['acc_drop'].values]
ax.barh(imp_df['feature'].values, imp_df['acc_drop'].values, color=colors)
ax.axvline(x=0, color='black', linestyle='--', alpha=0.3)
ax.set_xlabel('Accuracy Drop When Removed (positive = feature helps)')
ax.set_title('Leave-One-Out Feature Importance', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

## 5. Recursive Feature Elimination
Progressively remove least-important features.

In [ ]:
# RFE: start with all features, remove worst one each round
remaining_features = list(range(NEW_PHYS_DIM))
rfe_history = [{'n_features': len(remaining_features), 'acc': base_result['acc'],
                'wf1': base_result['wf1'], 'ri_f1': base_result['ri_f1'],
                'removed': 'None'}]

# Sort by importance (remove least important first)
feature_order = imp_df.sort_values('acc_drop', ascending=True)['feature'].tolist()

# Remove features one by one in order of least importance
n_rfe_steps = min(8, len(ALL_PHYS_NAMES) - 3)  # Keep at least 3 features
for step in range(n_rfe_steps):
    feat_to_remove = feature_order[step]
    feat_idx = ALL_PHYS_NAMES.index(feat_to_remove)
    remaining_features = [f for f in remaining_features if f != feat_idx]
    mask = np.zeros(NEW_PHYS_DIM, dtype=bool)
    mask[remaining_features] = True
    
    print(f"\nRFE Step {step+1}: Removing '{feat_to_remove}' → {len(remaining_features)} features left")
    r = run_fs_experiment(f'RFE-{len(remaining_features)}', ALL_SAMPLES, TARGET_BASIN,
        feature_mask=mask, epochs=RFE_EPOCHS, use_3d=USE_3D, use_env=USE_ENV, bs=BATCH_SIZE)
    rfe_history.append({'n_features': len(remaining_features), 'acc': r['acc'],
                        'wf1': r['wf1'], 'ri_f1': r['ri_f1'], 'removed': feat_to_remove})

rfe_df = pd.DataFrame(rfe_history)
display(rfe_df.round(4))

In [ ]:
# Visualise RFE curve
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric, label in [(axes[0],'acc','Accuracy'), (axes[1],'wf1','Weighted F1'), (axes[2],'ri_f1','RI F1')]:
    ax.plot(rfe_df['n_features'], rfe_df[metric], 'o-', linewidth=2, markersize=8)
    ax.set_xlabel('Number of Physics Features')
    ax.set_ylabel(label)
    ax.set_title(f'RFE: {label} vs Feature Count', fontweight='bold')
    ax.invert_xaxis()  # More features on left
    ax.grid(True, alpha=0.3)
    # Mark best point
    best_idx = rfe_df[metric].idxmax()
    ax.scatter(rfe_df.loc[best_idx,'n_features'], rfe_df.loc[best_idx,metric],
              s=200, c='red', zorder=5, marker='*')

plt.suptitle('Recursive Feature Elimination', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

## 6. Domain-Generalisation–Specific Feature Selection
Test whether some features hurt in-basin accuracy but *help* cross-basin transfer.

In [ ]:
# Compare: basin-specific features vs basin-invariant features
# Basin-specific: features correlated with basin identity (lon, lat, area-related)
# Basin-invariant: physics-based features (shear, coriolis, MPI)

# Define feature subsets
basin_invariant_idx = [i for i, name in enumerate(ALL_PHYS_NAMES)
                       if name in ['wind_shear', 'coriolis', 'coriolis_enhanced', 'mpi_proxy',
                                   'bl_moisture', 'current_int', 'shear', 'shear_int_interaction',
                                   'intensification_potential', 'favourable_shear', 'wnd_squared',
                                   'coriolis_x_wind', 'wind_pres_ratio', 'wind_pres_diff']]

basin_specific_idx = [i for i, name in enumerate(ALL_PHYS_NAMES)
                      if name in ['sst_anom', 'outflow_temp', 'steering', 'sst_proxy',
                                  'sst_bin', 'translation_proxy', 'abs_lat', 'pres_cubed',
                                  'mpi_deficit']]

subsets = [
    ('All features', None),
    ('Basin-invariant only', np.array([i in basin_invariant_idx for i in range(NEW_PHYS_DIM)])),
    ('Basin-specific only', np.array([i in basin_specific_idx for i in range(NEW_PHYS_DIM)])),
    ('Original 8 only', np.array([i < ORIG_PHYS_DIM for i in range(NEW_PHYS_DIM)])),
]

DG_RESULTS = []
for name, mask in subsets:
    n = int(mask.sum()) if mask is not None else NEW_PHYS_DIM
    print(f"\n{'='*60}")
    print(f"{name} ({n} features)")
    print(f"{'='*60}")
    r = run_fs_experiment(name, ALL_SAMPLES, TARGET_BASIN,
        feature_mask=mask, epochs=EPOCHS, use_3d=USE_3D, use_env=USE_ENV, bs=BATCH_SIZE)
    DG_RESULTS.append(r)

## 7. Final Recommendations

In [ ]:
print("═" * 80)
print("FEATURE SELECTION RESULTS")
print("═" * 80)

# DG comparison table
print(f"\n{'Subset':<25} {'#Feats':>6} {'Acc':>8} {'wF1':>8} {'RI-F1':>8}")
print("-" * 60)
for r in DG_RESULTS:
    print(f"{r['name']:<25} {r['n_feats']:>6} {r['acc']:>8.3f} {r['wf1']:>8.3f} {r['ri_f1']:>8.3f}")

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
names = [r['name'] for r in DG_RESULTS]
metrics = ['acc', 'wf1', 'ri_f1']
x = np.arange(len(names))
width = 0.25
for i, (m, label) in enumerate(zip(metrics, ['Accuracy', 'Weighted F1', 'RI F1'])):
    vals = [r[m] for r in DG_RESULTS]
    ax.bar(x + i*width, vals, width, label=label)
ax.set_xticks(x + width); ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_title(f'Feature Subset Comparison (Target: {TARGET_BASIN})', fontweight='bold')
ax.legend(); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

# RFE optimal point
best_rfe = rfe_df.loc[rfe_df['wf1'].idxmax()]

print("\n" + "═" * 80)
print("FINAL RECOMMENDATIONS")
print("═" * 80)
print(f"""
1. OPTIMAL FEATURE COUNT (from RFE):
   Best wF1 achieved with {int(best_rfe['n_features'])} features (wF1={best_rfe['wf1']:.3f})

2. FEATURE IMPORTANCE RANKING (top 5 most important):
""")
for i, row in imp_df.head(5).iterrows():
    print(f"   • {row['feature']}: acc_drop={row['acc_drop']:+.4f}")

print(f"""
3. DOMAIN GENERALISATION INSIGHT:
   Basin-invariant features vs basin-specific features comparison shows
   which set transfers better to unseen basins.

4. RECOMMENDED FEATURE SET:
   Keep the top features from importance ranking, prioritising:
   - Physics-based features (shear, coriolis, MPI)
   - Basin-invariant interactions (shear×intensity, coriolis×wind)
   Remove basin-specific features that may hurt generalisation.
""")
print("✅ Feature Selection analysis complete!")